# Centipede: where the time of training goes, on a Kaggle T4

This notebook runs `benchmarks/training_profile.py` from the public repository.
For each world count it builds the real pipeline from `configs/training.toml`,
runs a few short windows of real training, and reports the time of every part
of a step and of the update, the CPU's waits for the GPU, the most expensive GPU
kernels, the physics with and without a CUDA graph, the learning update at
several minibatch sizes, the work between cycles, memory, and a projection of
the whole configured run. The CPU backend is profiled the same way for
comparison, and the two GPUs once at the same time.

Before running, in the notebook's side panel:

- **Accelerator:** GPU T4 x2.
- **Internet:** on.

Then choose **Run all**. Everything takes roughly 30 to 45 minutes. The results
are saved in `/kaggle/working/profile` and zipped into
`/kaggle/working/profile.zip`, which can be downloaded from the notebook's
output panel. Each folder holds `report.txt` (what the cells print),
`profile.json` (every number), and the profiler traces, which open in
https://ui.perfetto.dev.

In [ ]:
# The machine: GPUs, driver, CPU cores, and Python version.
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!nproc
!python --version

In [ ]:
# Fetch the project and the two sibling packages it installs. Change BRANCH
# to profile another branch of Centipede.
OWNER = "https://github.com/DrippingSoup22"
BRANCH = "main"

!rm -rf /kaggle/working/centipede /kaggle/working/RL_lib /kaggle/working/MujocoReplay
!git clone --depth 1 --branch {BRANCH} {OWNER}/centipede-project.git /kaggle/working/centipede
!git clone --depth 1 {OWNER}/RL_lib.git /kaggle/working/RL_lib
!git clone --depth 1 {OWNER}/MujocoReplay.git /kaggle/working/MujocoReplay
%cd /kaggle/working/centipede
!git log -1 --format="Centipede commit %h: %s"
!git -C /kaggle/working/RL_lib log -1 --format="RL_lib commit %h: %s"
!git -C /kaggle/working/MujocoReplay log -1 --format="MujocoReplay commit %h: %s"

In [ ]:
# Install RL_lib (the learning algorithms) and MujocoReplay (which writes the
# replay recordings), then the package with the GPU backend. Kaggle
# already provides PyTorch with CUDA, so it is not downloaded again.
!pip install --quiet -e /kaggle/working/RL_lib -e /kaggle/working/MujocoReplay
!pip install --quiet -e ".[gpu]"

In [ ]:
# The versions in use, and the GPU as PyTorch and Warp see it.
import mujoco
import mujoco_warp
import torch
import warp

print("MuJoCo", mujoco.__version__, "| MuJoCo Warp", mujoco_warp.__version__)
print("Warp", warp.__version__, "| PyTorch", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(),
    "| compute capability",
    torch.cuda.get_device_capability(),
    "|",
    torch.cuda.device_count(),
    "visible",
)
assert torch.cuda.get_device_capability() >= (7, 0), "Choose the T4 accelerator"

In [ ]:
# The GPU at three world counts, with the configuration's other settings. The
# first world count also pays the kernel compilation of a fresh machine.
!python benchmarks/training_profile.py configs/training.toml     --worlds 256 1024 4096 --output /kaggle/working/profile/gpu

In [ ]:
# The CPU backend with the probe's settings, for comparison.
!python benchmarks/training_profile.py configs/probe.toml     --worlds 4 16 --output /kaggle/working/profile/cpu

In [ ]:
%%bash
# Both GPUs at once: two independent processes, each seeing one GPU, as two
# training runs would. Compare their speed with the 1,024-world profile above;
# the CPU cores are shared, so they may slow each other down.
cd /kaggle/working/centipede
for gpu in 0 1; do
    CUDA_VISIBLE_DEVICES=$gpu python benchmarks/training_profile.py         configs/training.toml --worlds 1024         --output /kaggle/working/profile/two_gpus_$gpu         > /kaggle/working/profile/two_gpus_$gpu.log 2>&1 &
done
wait
for gpu in 0 1; do
    echo "--- GPU $gpu"
    grep -A3 "Collecting and learning as training runs"         /kaggle/working/profile/two_gpus_$gpu.log
done

In [ ]:
# Keep everything in one file for download.
!cd /kaggle/working && zip -qr profile.zip profile && ls -lh profile.zip